# 03b · Irigat 2024, „Insulele verzi”
Clasificator pe indicii lunari 2024 (NDVI, NDMI, Sentinel-1 VV/VH) și ploaia ERA5-Land, antrenat pe etichetele din `data/out/irrigation_labels.gpkg`, doar pentru culturile de primăvară. Metodă de referință: Pageot et al. 2020, doi:10.3390/rs12183044. Validare pe blocuri de 5 km.

- Rezultate: `data/out/irrigation_2024.parquet` (`irrigated_prob`, `irrigated`) și `data/out/irrigation_metrics.json`.
- `irrigated_prob` e strat de hartă, **nu** factor în modelul 04: e calculat din aceiași indici de vară 2024 ca ținta.
- Dacă F1 pe blocuri e sub 0,75, stratul apare pe hartă ca „candidate”, nu ca „irigat”.

In [ ]:
import os, sys, json
if not os.path.exists('gisday_config.py') and os.path.exists('../gisday_config.py'):
    os.chdir('..')
sys.path.insert(0, os.getcwd())
from gisday_config import *
from gisday_lib import out_of_fold_probabilities, LGB_CLASSIFIER
import numpy as np, pandas as pd, geopandas as gpd, lightgbm as lgb
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score
parcels = gpd.read_parquet(OUT / 'parcels.parquet')
F = pd.read_parquet(OUT / 'parcel_features.parquet')
T = pd.read_parquet(OUT / 'parcel_target.parquet')[['parcel_id', 'is_arable', 'crop_group_2024']]
c = parcels.geometry.centroid
blocks = pd.DataFrame({'parcel_id': parcels.parcel_id, 'block': (c.x // 5000).astype(int).astype(str) + '_' + (c.y // 5000).astype(int).astype(str)})
D = F.merge(T, on='parcel_id').merge(blocks, on='parcel_id')
labels = gpd.read_file(OUT / 'irrigation_labels.gpkg', layer='etichete')[['parcel_id', 'irigat']].dropna(subset=['irigat'])
print('etichete completate:', len(labels), labels.irigat.value_counts().to_dict())
assert len(labels) >= 150, 'Sunt prea puține etichete: completează cel puțin 150 de parcele în irrigation_labels.gpkg'

In [ ]:
MONTHS = range(4, 11)
FEATURES = [f'{b}_{m:02d}' for b in ('ndvi', 'ndmi', 'vv', 'vh', 'rain') for m in MONTHS]
L = D.merge(labels, on='parcel_id')
assert L.crop_group_2024.eq('primavara').all(), 'Etichetele trebuie să fie doar pe culturi de primăvară'
X, y, groups = L[FEATURES], L['irigat'].astype(int), L['block']
p = out_of_fold_probabilities(X, y, groups)
metrics = dict(n_labels=len(L), f1=round(float(f1_score(y, p >= 0.5)), 3), auc=round(float(roc_auc_score(y, p)), 3),
               precision=round(float(precision_score(y, p >= 0.5)), 3), recall=round(float(recall_score(y, p >= 0.5)), 3))
metrics['layer_status'] = 'irigat' if metrics['f1'] >= 0.75 else 'candidate'
(OUT / 'irrigation_metrics.json').write_text(json.dumps(metrics, indent=1), encoding='utf-8')
print('validare pe blocuri de 5 km:', metrics)

In [ ]:
model = lgb.LGBMClassifier(**LGB_CLASSIFIER, random_state=2024).fit(X, y)
S = D[D.is_arable & D.crop_group_2024.eq('primavara')].copy()
S['irrigated_prob'] = model.predict_proba(S[FEATURES])[:, 1]
S['irrigated'] = S.irrigated_prob >= 0.5
S[['parcel_id', 'irrigated_prob', 'irrigated']].to_parquet(OUT / 'irrigation_2024.parquet')
area = parcels.set_index('parcel_id').area_ha.reindex(S.parcel_id).to_numpy()
print(f'culturi de primăvară: {len(S)} parcele, {area.sum():.0f} ha | irigate: {int(S.irrigated.sum())} parcele, {area[S.irrigated.to_numpy()].sum():.0f} ha')